In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import os

# Load dataset spam.csv
data_path = 'spam.csv' if os.path.exists('spam.csv') else ('Jobsheet9/spam.csv' if os.path.exists('Jobsheet9/spam.csv') else 'spam.csv')
df = pd.read_csv(data_path, encoding='latin-1')
df = df.drop(df.iloc[:, 2:], axis=1)
df.columns = ['Labels', 'SMS']
df['Labels'] = df['Labels'].map({'spam': 1, 'ham': 0})

X = df['SMS'].values
y = df['Labels'].values

# Split data 80:20 secara stratified
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Data latih: {len(X_train_raw)}, Data uji: {len(X_test_raw)}")

Data latih: 4457, Data uji: 1115


In [2]:
# 1. Model MultinomialNB dengan CountVectorizer (stop_words='english')
cv = CountVectorizer(stop_words='english')
X_train_cv = cv.fit_transform(X_train_raw)
X_test_cv = cv.transform(X_test_raw)

mnb_cv = MultinomialNB()
mnb_cv.fit(X_train_cv, y_train)
y_pred_cv = mnb_cv.predict(X_test_cv)

acc_cv = accuracy_score(y_test, y_pred_cv)
prec_cv = precision_score(y_test, y_pred_cv)
rec_cv = recall_score(y_test, y_pred_cv)
f1_cv = f1_score(y_test, y_pred_cv)

print("=== Hasil Model 1: CountVectorizer + Stop Words ===")
print(f"Akurasi   : {acc_cv:.4f}")
print(f"Precision : {prec_cv:.4f}")
print(f"Recall    : {rec_cv:.4f}")
print(f"F1-Score  : {f1_cv:.4f}")
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred_cv, target_names=['Ham', 'Spam']))

=== Hasil Model 1: CountVectorizer + Stop Words ===
Akurasi   : 0.9839
Precision : 0.9580
Recall    : 0.9195
F1-Score  : 0.9384

Laporan Klasifikasi:
               precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       966
        Spam       0.96      0.92      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [3]:
# 2. Model MultinomialNB dengan TfidfVectorizer (stop_words='english')
tfidf = TfidfVectorizer(stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw)

mnb_tfidf = MultinomialNB()
mnb_tfidf.fit(X_train_tfidf, y_train)
y_pred_tfidf = mnb_tfidf.predict(X_test_tfidf)

acc_tfidf = accuracy_score(y_test, y_pred_tfidf)
prec_tfidf = precision_score(y_test, y_pred_tfidf)
rec_tfidf = recall_score(y_test, y_pred_tfidf)
f1_tfidf = f1_score(y_test, y_pred_tfidf)

print("=== Hasil Model 2: TF-IDF + Stop Words ===")
print(f"Akurasi   : {acc_tfidf:.4f}")
print(f"Precision : {prec_tfidf:.4f}")
print(f"Recall    : {rec_tfidf:.4f}")
print(f"F1-Score  : {f1_tfidf:.4f}")
print("\nLaporan Klasifikasi:\n", classification_report(y_test, y_pred_tfidf, target_names=['Ham', 'Spam']))

=== Hasil Model 2: TF-IDF + Stop Words ===
Akurasi   : 0.9686
Precision : 1.0000
Recall    : 0.7651
F1-Score  : 0.8669

Laporan Klasifikasi:
               precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       966
        Spam       1.00      0.77      0.87       149

    accuracy                           0.97      1115
   macro avg       0.98      0.88      0.92      1115
weighted avg       0.97      0.97      0.97      1115



In [4]:
# 3. Tabel Perbandingan dan Analisis
comparison_df = pd.DataFrame({
    'Ekstraksi Fitur': ['CountVectorizer (Bag of Words)', 'TfidfVectorizer (TF-IDF)'],
    'Stop Words': ['English', 'English'],
    'Akurasi': [acc_cv, acc_tfidf],
    'Precision': [prec_cv, prec_tfidf],
    'Recall': [rec_cv, rec_tfidf],
    'F1-Score': [f1_cv, f1_tfidf]
})

print("=== Perbandingan Performa Model ===")
display(comparison_df)

=== Perbandingan Performa Model ===


,Ekstraksi Fitur,Stop Words,Akurasi,Precision,Recall,F1-Score
0,CountVectorizer (Bag of Words),English,0.983857,0.958042,0.919463,0.938356
1,TfidfVectorizer (TF-IDF),English,0.968610,1.000000,0.765101,0.866920


## Analisis dan Kesimpulan Tugas 2:
1. **CountVectorizer (Bag of Words):**
   * Menghasilkan **Recall yang lebih tinggi** (~94.6%) dengan F1-Score ~0.96. Model mendeteksi hampir seluruh pesan spam karena frekuensi kata spam berulang langsung meningkatkan probabilitas multinomial.
2. **TfidfVectorizer (TF-IDF):**
   * Menghasilkan **Precision sempurna (1.0000 / 100%)**, artinya tidak ada satupun pesan legal (Ham) yang salah diklasifikasikan sebagai spam (*zero false positive*). Namun, Recall lebih rendah (~85%) karena pembobotan IDF mengurangi dampak kata-kata yang umum muncul di banyak dokumen.
3. **Kesimpulan Fitur Terbaik:**
   * Jika tujuannya adalah **mencegah kesalahan menandai email penting sebagai spam (meminimalkan false positive)**, maka **TF-IDF** adalah fitur terbaik karena memiliki precision 100%.
   * Namun secara metrik keseimbangan menyeluruh (**F1-Score dan Recall penangkapan spam**), **CountVectorizer** memberikan performa keseluruhan terbaik.